In [2]:
thurgau = {
    0: {'name': 'Romanshorn', 'population': 11556, 'latitude': 47.56586, 'longitude': 9.37869},
    1: {'name': 'Amriswil', 'population': 14313, 'latitude': 47.54814, 'longitude': 9.30327},
    2: {'name': 'Arbon', 'population': 15459, 'latitude': 47.51360, 'longitude': 9.42999},
    3: {'name': 'Weinfelden', 'population': 11893, 'latitude': 47.56638, 'longitude': 9.10588},
    4: {'name': 'Frauenfeld', 'population': 26093, 'latitude': 47.55856, 'longitude': 8.89685},
    5: {'name': 'Kreuzlingen', 'population': 22788, 'latitude': 47.645837,'longitude': 9.178608},
    6: {'name': 'Egnach', 'population': 4897, 'latitude': 47.54565, 'longitude': 9.37864},
}

In [3]:
from tqdm.auto import tqdm

def linear_search(dataset, attribute, query):
    for key, document in tqdm(dataset.items()):
        if document[attribute] == query:
            yield document

In [4]:
for document in linear_search(thurgau, 'name', 'Romanshorn'): print(document)

  0%|          | 0/7 [00:00<?, ?it/s]

{'name': 'Romanshorn', 'population': 11556, 'latitude': 47.56586, 'longitude': 9.37869}


In [5]:
from pathlib import Path
from zipfile import ZipFile
from io import TextIOWrapper
import csv

def read_geonames_database(basename):
    filename = (Path('data') / basename).with_suffix(".zip")
    fieldnames = ['geonameid','name','asciiname', 'alternatenames', 'latitude', 'longitude', 'feature class', 'feature code', 'country code', 'cc2', 'admin1 code', 'admin2 code', 'admin3 code', 'admin4 code', 'population', 'elevation', 'dem', 'timezone', 'modification date']
 
    result = dict()
    with ZipFile(filename) as zip:
        stream = zip.open(basename + ".txt")
        reader = csv.DictReader(TextIOWrapper(stream, 'utf-8'), delimiter='\t', fieldnames=fieldnames)
        for row in tqdm(reader):
            result[int(row['geonameid'])] = row

    return result

cities500 = read_geonames_database('cities500')
allplaces = read_geonames_database('allCountries')


0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [6]:
print(cities500[2658985])

{'geonameid': '2658985', 'name': 'Romanshorn', 'asciiname': 'Romanshorn', 'alternatenames': 'Romanshorn,Romanskhorn,luo man si huo en,Романсхорн,羅曼斯霍恩', 'latitude': '47.56586', 'longitude': '9.37869', 'feature class': 'P', 'feature code': 'PPLA3', 'country code': 'CH', 'cc2': '', 'admin1 code': 'TG', 'admin2 code': '2011', 'admin3 code': '4436', 'admin4 code': '', 'population': '8956', 'elevation': '', 'dem': '401', 'timezone': 'Europe/Zurich', 'modification date': '2013-04-02'}


In [8]:
%%time
for place in linear_search(allplaces, 'name', 'Romanshorn'): print(place)

  0%|          | 0/12410889 [00:00<?, ?it/s]

{'geonameid': '2658985', 'name': 'Romanshorn', 'asciiname': 'Romanshorn', 'alternatenames': 'Romanshorn,Romanskhorn,luo man si huo en,Романсхорн,羅曼斯霍恩', 'latitude': '47.56586', 'longitude': '9.37869', 'feature class': 'P', 'feature code': 'PPLA3', 'country code': 'CH', 'cc2': '', 'admin1 code': 'TG', 'admin2 code': '2011', 'admin3 code': '4436', 'admin4 code': '', 'population': '8956', 'elevation': '', 'dem': '401', 'timezone': 'Europe/Zurich', 'modification date': '2013-04-02'}
{'geonameid': '7286940', 'name': 'Romanshorn', 'asciiname': 'Romanshorn', 'alternatenames': 'CH4436', 'latitude': '47.56354', 'longitude': '9.35639', 'feature class': 'A', 'feature code': 'ADM3', 'country code': 'CH', 'cc2': '', 'admin1 code': 'TG', 'admin2 code': '2011', 'admin3 code': '4436', 'admin4 code': '', 'population': '11269', 'elevation': '', 'dem': '428', 'timezone': 'Europe/Zurich', 'modification date': '2021-12-03'}
{'geonameid': '11963382', 'name': 'Romanshorn', 'asciiname': 'Romanshorn', 'alterna

In [9]:
def build_attribute_index(dataset, attribute):
    index = dict()
    for key, document in tqdm(dataset.items()):
        value = document[attribute]
        index.setdefault(value, set()).add(key)
    return index

name_index = build_attribute_index(thurgau, 'name')
name_index

  0%|          | 0/7 [00:00<?, ?it/s]

{'Romanshorn': {0},
 'Amriswil': {1},
 'Arbon': {2},
 'Weinfelden': {3},
 'Frauenfeld': {4},
 'Kreuzlingen': {5},
 'Egnach': {6}}

In [10]:
def query_index(index, dataset, query):
    for key in index.get(query, set()):
        yield dataset[key]

for document in query_index(name_index, thurgau, "Egnach"): print(document)

{'name': 'Egnach', 'population': 4897, 'latitude': 47.54565, 'longitude': 9.37864}


In [11]:
%%time
name_index = build_attribute_index(allplaces, 'name')


  0%|          | 0/12410889 [00:00<?, ?it/s]

CPU times: user 23.9 s, sys: 1min 14s, total: 1min 38s
Wall time: 2min 54s


In [15]:
print(len(name_index['London']))

65


In [70]:
%%time
for document in query_index(name_index, allplaces, "Egnach"): print(document)

{'geonameid': '2660932', 'name': 'Egnach', 'asciiname': 'Egnach', 'alternatenames': 'Egnach,Ehgnakh,ai ge na he,Эгнах,埃格納赫', 'latitude': '47.54268', 'longitude': '9.38048', 'feature class': 'P', 'feature code': 'PPL', 'country code': 'CH', 'cc2': '', 'admin1 code': 'TG', 'admin2 code': '2011', 'admin3 code': '4411', 'admin4 code': '', 'population': '4179', 'elevation': '', 'dem': '405', 'timezone': 'Europe/Zurich', 'modification date': '2015-09-06'}
CPU times: user 212 μs, sys: 543 μs, total: 755 μs
Wall time: 1.39 ms
